# ML Capstone — Review 2: Classification Part B
- **Dataset:** RTA Dataset (Addis Ababa, 2017–20), 12316 rows × 32 cols
- **Target:** `Accident_severity` (3 classes: Fatal / Serious / Slight)
- **Algorithms (6–10):** Random Forest, AdaBoost, Gradient Boosting, Bagging, MLP
- **Pipeline:** sklearn `Pipeline` + `ColumnTransformer` throughout
- **Tuning:** `GridSearchCV` (5-fold stratified) for all models


## 1. Imports & Setup

In [1]:
import warnings, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, BaggingClassifier
)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.base import clone
warnings.filterwarnings('ignore')
SEED = 42
print('Imports OK')


Imports OK


## 2. Data Loading & EDA
- Load CSV, drop rows where target is NaN
- Check shape, missing %, class distribution


In [2]:
DATA = '/Users/abhinavreddy/Documents/SEM_5/ML/ML_proj/Classification/RTA Dataset.csv'
df = pd.read_csv(DATA)
df = df.dropna(subset=['Accident_severity'])
TARGET = 'Accident_severity'
print('Shape:', df.shape)
print('\nMissing % per column:')
print((df.isnull().sum() / len(df) * 100).round(2).sort_values(ascending=False).head(10))
print('\nClass distribution:')
print(df[TARGET].value_counts())


Shape: (12316, 32)

Missing % per column:
Defect_of_vehicle          35.95
Service_year_of_vehicle    31.89
Work_of_casuality          25.97
Fitness_of_casuality       21.39
Type_of_vehicle             7.71
Types_of_Junction           7.20
Driving_experience          6.73
Educational_level           6.02
Vehicle_driver_relation     4.70
Owner_of_vehicle            3.91
dtype: float64

Class distribution:
Accident_severity
Slight Injury     10415
Serious Injury     1743
Fatal injury        158
Name: count, dtype: int64


### Target & Feature Distributions
- Class imbalance: Slight Injury ~85%, Serious ~14%, Fatal ~1%


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452','#55A868'])
axes[0].set_title('Target Class Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
# Correlation heatmap on numeric cols
num_df = df.select_dtypes(include='number')
sns.heatmap(num_df.corr(), ax=axes[1], cmap='coolwarm', annot=False)
axes[1].set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('eda_classification_partb.png', dpi=80, bbox_inches='tight')
plt.show()
print('EDA plots saved.')


EDA plots saved.


## 3. Preprocessing Pipeline
- Numeric: `SimpleImputer(median)` → `StandardScaler`
- Categorical: `SimpleImputer(most_frequent)` → `OrdinalEncoder`
- `ColumnTransformer` wraps both; embedded in every model `Pipeline`


In [4]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET]

# encode target
le = LabelEncoder()
y_enc = le.fit_transform(y)
classes = le.classes_
print('Classes:', classes)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=SEED, stratify=y_enc
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')


Classes: ['Fatal injury' 'Serious Injury' 'Slight Injury']
Train: (9852, 31), Test: (2464, 31)


## 4. Model Training with Hyperparameter Tuning
- Each model wrapped in `Pipeline(preprocessor → classifier)`
- `GridSearchCV` with 5-fold stratified CV
- Metrics: Accuracy, Precision (w), Recall (w), F1 (w), ROC-AUC (OvR)


In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate(name, pipeline, X_tr, X_te, y_tr, y_te, classes):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)
    y_bin  = label_binarize(y_te, classes=range(len(classes)))
    roc    = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
    return {
        'Model': name,
        'Accuracy':  round(accuracy_score(y_te, y_pred), 4),
        'Precision': round(precision_score(y_te, y_pred, average='weighted', zero_division=0), 4),
        'Recall':    round(recall_score(y_te, y_pred, average='weighted', zero_division=0), 4),
        'F1':        round(f1_score(y_te, y_pred, average='weighted', zero_division=0), 4),
        'ROC-AUC':   round(roc, 4),
    }

results_B = []
best_estimators = {}


### Model 6: Random Forest
- Ensemble of decision trees (bagging + random features)

In [6]:
rf_pipe = Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(random_state=SEED))])
rf_grid = GridSearchCV(
    rf_pipe,
    {'clf__n_estimators': [100, 200], 'clf__max_depth': [None, 10], 'clf__min_samples_split': [2, 5]},
    cv=cv, scoring='f1_weighted', n_jobs=-1
)
rf_grid.fit(X_train, y_train)
best_estimators['Random Forest'] = rf_grid.best_estimator_
print('Best params:', rf_grid.best_params_)
r = evaluate('Random Forest', rf_grid.best_estimator_, X_train, X_test, y_train, y_test, classes)
results_B.append(r)
print(r)


Best params: {'clf__max_depth': None, 'clf__min_samples_split': 2, 'clf__n_estimators': 200}


{'Model': 'Random Forest', 'Accuracy': 0.847, 'Precision': 0.798, 'Recall': 0.847, 'F1': 0.7791, 'ROC-AUC': np.float64(0.6866)}


### Model 7: AdaBoost
- Sequential boosting; weights misclassified samples higher

In [7]:
ada_pipe = Pipeline([('pre', preprocessor), ('clf', AdaBoostClassifier(random_state=SEED, algorithm='SAMME'))])
ada_grid = GridSearchCV(
    ada_pipe,
    {'clf__n_estimators': [50, 100], 'clf__learning_rate': [0.5, 1.0]},
    cv=cv, scoring='f1_weighted', n_jobs=-1
)
ada_grid.fit(X_train, y_train)
best_estimators['AdaBoost'] = ada_grid.best_estimator_
print('Best params:', ada_grid.best_params_)
r = evaluate('AdaBoost', ada_grid.best_estimator_, X_train, X_test, y_train, y_test, classes)
results_B.append(r)
print(r)


/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorith

/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorith

/opt/anaconda3/lib/python3.13/site-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


Best params: {'clf__learning_rate': 0.5, 'clf__n_estimators': 50}


{'Model': 'AdaBoost', 'Accuracy': 0.8458, 'Precision': 0.7153, 'Recall': 0.8458, 'F1': 0.7751, 'ROC-AUC': np.float64(0.6222)}


### Model 8: Gradient Boosting
- Fits shallow trees to pseudo-residuals sequentially

In [8]:
gb_pipe = Pipeline([('pre', preprocessor), ('clf', GradientBoostingClassifier(random_state=SEED))])
gb_grid = GridSearchCV(
    gb_pipe,
    {'clf__n_estimators': [100, 200], 'clf__learning_rate': [0.05, 0.1], 'clf__max_depth': [3, 5]},
    cv=cv, scoring='f1_weighted', n_jobs=-1
)
gb_grid.fit(X_train, y_train)
best_estimators['Gradient Boosting'] = gb_grid.best_estimator_
print('Best params:', gb_grid.best_params_)
r = evaluate('Gradient Boosting', gb_grid.best_estimator_, X_train, X_test, y_train, y_test, classes)
results_B.append(r)
print(r)


Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 5, 'clf__n_estimators': 200}


{'Model': 'Gradient Boosting', 'Accuracy': 0.8519, 'Precision': 0.8295, 'Recall': 0.8519, 'F1': 0.809, 'ROC-AUC': np.float64(0.7697)}


### Model 9: Bagging Classifier
- Bootstrap aggregation over base decision trees

In [9]:
bag_pipe = Pipeline([('pre', preprocessor), ('clf', BaggingClassifier(random_state=SEED))])
bag_grid = GridSearchCV(
    bag_pipe,
    {'clf__n_estimators': [10, 50], 'clf__max_samples': [0.8, 1.0]},
    cv=cv, scoring='f1_weighted', n_jobs=-1
)
bag_grid.fit(X_train, y_train)
best_estimators['Bagging'] = bag_grid.best_estimator_
print('Best params:', bag_grid.best_params_)
r = evaluate('Bagging', bag_grid.best_estimator_, X_train, X_test, y_train, y_test, classes)
results_B.append(r)
print(r)


Best params: {'clf__max_samples': 1.0, 'clf__n_estimators': 50}


{'Model': 'Bagging', 'Accuracy': 0.8506, 'Precision': 0.8179, 'Recall': 0.8506, 'F1': 0.798, 'ROC-AUC': np.float64(0.7115)}


### Model 10: MLP Neural Network
- Feedforward net; Adam optimizer; ReLU hidden layers

In [10]:
mlp_pipe = Pipeline([('pre', preprocessor), ('clf', MLPClassifier(random_state=SEED, max_iter=300))])
mlp_grid = GridSearchCV(
    mlp_pipe,
    {'clf__hidden_layer_sizes': [(64,), (128, 64)], 'clf__alpha': [0.0001, 0.001]},
    cv=cv, scoring='f1_weighted', n_jobs=-1
)
mlp_grid.fit(X_train, y_train)
best_estimators['MLP'] = mlp_grid.best_estimator_
print('Best params:', mlp_grid.best_params_)
r = evaluate('MLP Neural Net', mlp_grid.best_estimator_, X_train, X_test, y_train, y_test, classes)
results_B.append(r)
print(r)


Best params: {'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (128, 64)}


{'Model': 'MLP Neural Net', 'Accuracy': 0.8409, 'Precision': 0.7436, 'Recall': 0.8409, 'F1': 0.7736, 'ROC-AUC': np.float64(0.5376)}


## 5. Confusion Matrices (Part B Models)
- Grid of 5 confusion matrices for all Part B classifiers


In [11]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, (name, est) in zip(axes, best_estimators.items()):
    y_pred = est.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(name)
for ax in axes[len(best_estimators):]:
    ax.set_visible(False)
plt.suptitle('Confusion Matrices — Part B Classifiers', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices_partb.png', dpi=80, bbox_inches='tight')
plt.show()


## 6. Consolidated 10-Algorithm Comparison Table (Rubric A2)
- Part A results (Algorithms 1–5) hard-coded from tuned runs
- Part B results (Algorithms 6–10) from GridSearchCV above


In [12]:
# Part A results from Classification folder notebooks (tuned)
results_A = [
    {'Model': 'Logistic Regression', 'Accuracy': 0.8458, 'Precision': 0.7893, 'Recall': 0.8458, 'F1': 0.7751, 'ROC-AUC': 0.5801},
    {'Model': 'KNN',                 'Accuracy': 0.8377, 'Precision': 0.7784, 'Recall': 0.8377, 'F1': 0.7753, 'ROC-AUC': 0.5615},
    {'Model': 'Naive Bayes',          'Accuracy': 0.8141, 'Precision': 0.7617, 'Recall': 0.8141, 'F1': 0.7648, 'ROC-AUC': 0.5648},
    {'Model': 'Decision Tree',        'Accuracy': 0.8429, 'Precision': 0.7959, 'Recall': 0.8429, 'F1': 0.7959, 'ROC-AUC': 0.6175},
    {'Model': 'SVC',                  'Accuracy': 0.8287, 'Precision': 0.7939, 'Recall': 0.8287, 'F1': 0.7883, 'ROC-AUC': 0.5734},
]
all_results = results_A + results_B
df_compare = pd.DataFrame(all_results)
df_compare.index = range(1, 11)
df_compare.index.name = 'Rank'
df_compare_sorted = df_compare.sort_values('F1', ascending=False).reset_index(drop=True)
df_compare_sorted.index = range(1, 11)
df_compare_sorted.index.name = 'Rank'
print(df_compare_sorted.to_string())


                    Model  Accuracy  Precision  Recall      F1  ROC-AUC
Rank                                                                   
1       Gradient Boosting    0.8519     0.8295  0.8519  0.8090   0.7697
2                 Bagging    0.8506     0.8179  0.8506  0.7980   0.7115
3           Decision Tree    0.8429     0.7959  0.8429  0.7959   0.6175
4                     SVC    0.8287     0.7939  0.8287  0.7883   0.5734
5           Random Forest    0.8470     0.7980  0.8470  0.7791   0.6866
6                     KNN    0.8377     0.7784  0.8377  0.7753   0.5615
7     Logistic Regression    0.8458     0.7893  0.8458  0.7751   0.5801
8                AdaBoost    0.8458     0.7153  0.8458  0.7751   0.6222
9          MLP Neural Net    0.8409     0.7436  0.8409  0.7736   0.5376
10            Naive Bayes    0.8141     0.7617  0.8141  0.7648   0.5648


## 7. Final Model Selection & Tuning Improvement (Rubric A3)
- Best model selected based on F1 (weighted) — primary metric
- Shows baseline (default params) vs tuned F1 delta


In [13]:
# Baseline Gradient Boosting (default params)
gb_baseline = Pipeline([('pre', preprocessor), ('clf', GradientBoostingClassifier(random_state=SEED))])
gb_baseline.fit(X_train, y_train)
y_pred_base = gb_baseline.predict(X_test)
f1_base = f1_score(y_test, y_pred_base, average='weighted')

# Tuned Gradient Boosting
f1_tuned = next(r['F1'] for r in results_B if 'Gradient' in r['Model'])

print('=' * 50)
print('BEST MODEL: Gradient Boosting Classifier')
print('=' * 50)
print(f'  Baseline F1 (default params): {f1_base:.4f}')
print(f'  Tuned F1   (GridSearchCV):    {f1_tuned:.4f}')
print(f'  Improvement:                  +{f1_tuned - f1_base:.4f}')
print()
print('Best hyperparameters:')
print(gb_grid.best_params_)


BEST MODEL: Gradient Boosting Classifier
  Baseline F1 (default params): 0.7875
  Tuned F1   (GridSearchCV):    0.8090
  Improvement:                  +0.0215

Best hyperparameters:
{'clf__learning_rate': 0.1, 'clf__max_depth': 5, 'clf__n_estimators': 200}


### Feature Importance (Best Model)
- Top 15 features from tuned Gradient Boosting

In [14]:
best_gb = best_estimators['Gradient Boosting']
feat_names = num_cols + cat_cols
importances = best_gb.named_steps['clf'].feature_importances_
feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances — Gradient Boosting')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance_gb.png', dpi=80, bbox_inches='tight')
plt.show()
